# Whisper large-v3-turbo — DIMER ASR tutorial

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.0  
**Capability:** multilingual automatic speech recognition using the pinned OpenAI Whisper large-v3-turbo weights

This notebook is the executable reference path for the repository capability. It exercises the repository's public pipeline API rather than reimplementing model inference. The default sample is demonstration evidence, not a production-quality or benchmark claim.

**Learning objectives:** bootstrap the repository in a fresh runtime, resolve the immutable upstream model revision, validate a public/default input, run the supported task, inspect task-appropriate outputs, exercise an optional BYOD path, and export machine-readable outputs plus provenance.


## Prerequisites

Run in a fresh supported runtime. The default path supports CPU and uses CUDA automatically when available. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.


## 1. Bootstrap the repository and pinned runtime

When the notebook is opened without a repository checkout, this cell clones the repository. Released notebooks default to `main`; automated candidate validation can set `DIMER_TUTORIAL_REF` to an immutable commit or review branch. The repository is installed as a regular (non-editable) package so it is importable in this same runtime; an editable install would only become importable after a restart. Model-facing dependencies are directly pinned. If installation replaces an already imported core package, the cell fails with a restart instruction rather than continuing with mixed versions.

In [ ]:
import importlib
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/kurtvalcorza/whisper-asr-pipeline.git'
REPO_NAME = 'whisper-asr-pipeline'
REPO_REF = os.environ.get('DIMER_TUTORIAL_REF', 'main')
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    checkout = ROOT / REPO_NAME
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '-q', REPO_URL, str(checkout)], check=True)
    if REPO_REF != 'main':
        subprocess.run(['git', '-C', str(checkout), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', '-C', str(checkout), 'checkout', '-q', 'main'], check=True)
        subprocess.run(['git', '-C', str(checkout), 'pull', '--ff-only', '-q', 'origin', 'main'], check=True)
    os.chdir(checkout)
    ROOT = Path.cwd()

if not SKIP_INSTALL:
    tracked = {'torch': 'torch', 'transformers': 'transformers', 'numpy': 'numpy'}
    # Distribution versions of core packages that are already imported, captured before
    # installation. Metadata is compared with metadata afterwards: torch.__version__ carries a
    # local build label (for example 2.6.0+cu124) that the distribution version omits.
    def _installed_version(distribution):
        try:
            return importlib.metadata.version(distribution)
        except importlib.metadata.PackageNotFoundError:
            return None
    loaded = {distribution: _installed_version(distribution) for distribution, module in tracked.items() if module in sys.modules}
    # Non-editable install: an editable (.pth) install is not importable until the
    # interpreter restarts, which a fresh hosted runtime cannot do mid-notebook.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'{ROOT}[tutorial]'], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

REPO_SHA = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
import platform, torch, transformers
print({'repository': str(ROOT), 'repository_revision': REPO_SHA, 'requested_ref': REPO_REF, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Load a public sample or optional BYOD

The default path loads the public `hf-internal-testing/librispeech_asr_dummy` sample and its reference text, decoding the stored audio bytes with the pinned `soundfile` dependency into a float32 waveform plus sampling rate. BYOD is optional and disabled by default; expected BYOD input is a local audio file readable by the ASR stack.

In [ ]:
import io

import soundfile as sf
from datasets import Audio, load_dataset

USE_BYOD = False
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    audio_input = next(iter(uploaded))
    reference = None
else:
    # The audio bytes are decoded with the pinned soundfile dependency. The datasets audio
    # feature would decode through torchcodec/FFmpeg, which this runtime does not pin.
    ds = load_dataset('hf-internal-testing/librispeech_asr_dummy', 'clean', split='validation')
    ds = ds.cast_column('audio', Audio(decode=False))
    row = ds[0]
    waveform, sampling_rate = sf.read(io.BytesIO(row['audio']['bytes']), dtype='float32')
    if waveform.ndim > 1:
        waveform = waveform.mean(axis=1)
    audio_input = {'array': waveform, 'sampling_rate': sampling_rate}
    reference = row['text']
print({'sample_type': 'BYOD' if USE_BYOD else 'public LibriSpeech dummy', 'has_reference': reference is not None, 'seconds': None if USE_BYOD else round(len(waveform) / sampling_rate, 2)})

## 3. Resolve the pinned model

The public API pins the exact upstream revision and refuses remote model code. The effective model identity is printed before inference.

In [ ]:
from whisper_asr_pipeline import MODEL_ID, MODEL_REVISION, WhisperASRPipeline, word_error_rate
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION})
pipe = WhisperASRPipeline.from_pretrained()
print({'device': pipe.device})

## 4. Run ASR and evaluate when ground truth exists

The pipeline returns normalized text plus provenance fields. WER is computed only when a reference transcript is available and is tutorial evidence, not a fleet benchmark.

In [ ]:
result = pipe.transcribe(audio_input, language='en', task='transcribe')
print(result['text'])
metrics = {}
if reference is not None:
    metrics['word_error_rate'] = word_error_rate(reference, result['text'])
print(metrics)

## 5. Export outputs and provenance

Machine-readable JSON preserves the transcript, measured tutorial metric, repository revision, model identifier, immutable model revision, and runtime identity.

In [ ]:
import json
os.makedirs('outputs', exist_ok=True)
payload = {
    'prediction': result,
    'metrics': metrics,
    'repository_revision': REPO_SHA,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
    'sample': 'BYOD' if USE_BYOD else 'hf-internal-testing/librispeech_asr_dummy',
}
with open('outputs/whisper_asr_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print('outputs/whisper_asr_result.json')

## Interpretation and limits

The ASR text is a model-generated transcript. WER, when shown, is tied to the single demonstrated reference and must not be generalized to other languages, speakers, domains, or capture conditions. The pipeline provides no diarization, speaker identity, biometric inference, or calibrated transcript-confidence threshold.

Successful execution proves that the recorded repository revision can acquire the pinned model, validate the demonstrated input, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

## References

- Repository README: `../README.md`
- Repository model card: `../MODEL_CARD.md`
- Upstream model: https://huggingface.co/openai/whisper-large-v3-turbo
- Whisper paper: https://arxiv.org/abs/2212.04356
